In [0]:
%sql
--Task 1: Channel performance

-- Nomsa needs the revenue split by channel for FY2024 and FY2025 side by side, with the growth rate between them. One row per channel, three figures on each. FY2024 is orders placed on or before 30 June 2024. FY2025 is everything after that. 

WITH draft_revenue AS 
(
    SELECT 
        channel,
        ROUND(SUM(
                CASE
                    WHEN order_date BETWEEN '2023-07-01' AND '2024-06-30'
                    THEN line_revenue
                    ELSE 0
                END
                  ),2) AS 2024_revenue,
        ROUND(SUM(
                CASE
                    WHEN order_date BETWEEN '2024-07-01' AND '2025-06-30'
                    THEN line_revenue
                    ELSE 0
                END
                 ),2) AS 2025_revenue
    FROM sizwe_sales_orders so
    JOIN sizwe_sales_lines sl
        ON so.order_id = sl.order_id
    GROUP BY channel
)
SELECT 
    channel,
    2024_revenue,
    2025_revenue,
    ROUND(((2025_revenue - 2024_revenue) / 2024_revenue * 100), 2) AS percent_growth
FROM draft_revenue


channel,2024_revenue,2025_revenue,percent_growth
Wholesale,3.0816601232E8,3.3542632502E8,8.85
Subscription,1.3283398515E8,1.4457369833E8,8.84


In [0]:
%sql
-- Task 2 - Where the profit actually comes from

-- For each product line, return revenue, cost of goods, gross profit and gross margin percentage. Sort by gross profit. In one or two sentences under your query: which line earns the most profit, and is it the same line that earns the highest margin?

SELECT 
    product_line,
    ROUND(SUM(line_revenue),2) AS p_line_revenue,
    ROUND(SUM(cogs),2) AS p_line_cogs,
    ROUND((SUM(line_revenue) - SUM(cogs)),2) AS gross_profit,
    ROUND(((SUM(line_revenue) - SUM(cogs)) / SUM(line_revenue) * 100),2) AS gross_margin_percent
FROM sizwe_products p 
JOIN sizwe_sales_lines sl
    ON p.sku = sl.sku
GROUP BY product_line
ORDER BY 4 DESC

-- Answer:
-- The most profitable product line is House Blend. However, it's Single Origin that has the highest gross margin percentage.


product_line,p_line_revenue,p_line_cogs,gross_profit,gross_margin_percent
House Blend,2.795309489E8,1.723822023E8,1.071487466E8,38.33
Single Origin,1.7011261961E8,9.479981874E7,7.531280087E7,44.27
Bulk Commercial,2.7183883101E8,2.1773595153E8,5.410287948E7,19.9
Capsules,1.0887507622E8,6.651657155E7,4.235850467E7,38.91
Decaf,9.064254508E7,5.816754701E7,3.247499807E7,35.83


In [0]:
%sql
-- Task 3 - The collections problem

-- Nomsa suspects the business is being paid more slowly than it used to be, and wants to know where the problem sits. For each channel, return the number of invoices, how many have been settled, how many are still open, the average number of days taken to settle, and the total value still outstanding. In two or three sentences: what does this tell you, and which channel would you investigate first?

SELECT 
    c.channel,
    COUNT(invoice_no) AS num_invoices,
    SUM(CASE
            WHEN invoice_amount = amount_settled THEN 1
            ELSE 0
        END) AS num_invoices_settled,
    SUM(CASE
            WHEN invoice_amount > amount_settled THEN 1
            ELSE 0
        END) AS num_invoices_unsettled,
    CEILING(AVG(DATEDIFF(DAY, invoice_date, settlement_date))) AS avg_days_settle,
    ROUND(SUM(invoice_amount - amount_settled),2) AS total_unsettled
FROM sizwe_customers c
JOIN sizwe_ar_ledger al
    ON c.customer_id = al.customer_id
GROUP BY c.channel

-- Answer: One can see that the major collections problem lies with the wholesale channel. Its unsettled invoices is significantly higher than the subscription channel. Likewise, the wholesale channel takes much longer to settle (69 vs 4 days) and the total unsettled amount is much higher than the subscription channel (~R87m vs ~R7.5m).


channel,num_invoices,num_invoices_settled,num_invoices_unsettled,avg_days_settle,total_unsettled
Wholesale,28944,25490,3454,69,8.78482272E7
Subscription,19200,19156,44,4,751178.89


In [0]:
%sql
-- Task 4 - The ten accounts that matter most

-- Bongani wants a one-page list of the ten largest customers by FY2025 revenue, showing the trading name, segment, province, the sales representative who owns the account, and the revenue. Not every customer has a sales representative. Make sure your query does not silently drop anyone. 

SELECT 
    trading_name,
    segment,
    province,
    rep_name,
    ROUND(SUM(CASE
            WHEN order_date BETWEEN '2024-07-01' AND '2025-06-30' THEN line_revenue
            ELSE 0
        END
        ),2) AS total_revenue_2025
FROM sizwe_customers c
JOIN sizwe_sales_orders so 
    ON c.customer_id = so.customer_id
JOIN sizwe_sales_lines sl
    ON so.order_id = sl.order_id
LEFT JOIN sizwe_rep_visits rv --Prevent silent dropping of customers without rep visits (i.e. they might have the highest total revenue without having a sales rep assigned to them)
    ON c.customer_id = rv.customer_id
LEFT JOIN sizwe_reps r --Prevent silent dropping of customers without rep visits, same logic as above
    ON r.rep_id = rv.rep_id
GROUP BY trading_name, segment, province, rep_name
ORDER BY 5 DESC
LIMIT 10


trading_name,segment,province,rep_name,total_revenue_2025
Halvane Retail Group,Grocery Chain,Western Cape,Kagiso Mkhize,7.2959999791E8
Highveld Deli 85,Restaurant Group,Free State,Karabo Mokoena,5.070300495E7
Table Bay Foods 280,Independent Cafe,KwaZulu-Natal,Karabo van Wyk,4.6671492E7
Boland Bakery 91,Independent Cafe,Free State,Karabo Zwane,4.332186181E7
Bloom Coffee House 275,Hotel Group,Gauteng,Karabo Zwane,4.255661058E7
Midrand Market 293,Independent Cafe,KwaZulu-Natal,Chantal Ismail,3.846468262E7
Kalahari Kitchen 42,Independent Cafe,Gauteng,Chantal Ismail,3.7134772E7
Ridgeway Foods 308,Independent Cafe,KwaZulu-Natal,Chantal Adams,3.712888592E7
Vaal Trading Co 256,Independent Cafe,North West,Karabo Mokoena,3.4635241E7
Corner Perk 34,Independent Cafe,Limpopo,Chantal Ismail,3.220382232E7


In [0]:
%sql
-- Task 5 - A commercial gap

-- Trade promotions are how Sizwe supports its wholesale accounts. Bongani thinks some sizeable customers have been getting nothing. Return the fifteen highest-revenue Wholesale customers who have never received a trade promotion, with their segment and province. One sentence: what would you recommend doing about the top three?

SELECT
    c.trading_name,
    c.segment,
    c.province,
    ROUND(SUM(sl.line_revenue),2) AS customer_total_revenue
FROM sizwe_customers c
JOIN sizwe_sales_orders so
    ON c.customer_id = so.customer_id
JOIN sizwe_sales_lines sl
    ON so.order_id = sl.order_id
LEFT JOIN sizwe_trade_promotions tp ---Not inner join, because we want to see which customers have not been offered a promotion
    ON c.customer_id = tp.customer_id
WHERE c.channel = 'Wholesale'
    AND tp.customer_id IS NULL
GROUP BY c.trading_name, c.segment, c.province
ORDER BY 4 DESC
LIMIT 15

-- Answer: I would get the Gauteng and KZN sales reps to telephonically reach these customers and offer them a trade promotion, as their total revenue is quite sizeable and we don't want to lose them to a competitor who might offer them a discount.




trading_name,segment,province,customer_total_revenue
Umhlanga Fresh 108,Restaurant Group,Gauteng,6224772.37
Ridgeway Foods 308,Independent Cafe,KwaZulu-Natal,4448744.72
Kalahari Kitchen 42,Independent Cafe,Gauteng,4447225.01
Midrand Market 293,Independent Cafe,KwaZulu-Natal,3887087.18
Bloom Coffee House 275,Hotel Group,Gauteng,3719361.46
Corner Perk 34,Independent Cafe,Limpopo,3431735.39
Highveld Deli 28,Independent Cafe,Gauteng,3421285.02
Bloom Coffee House 306,Restaurant Group,Gauteng,3397003.31
Kalahari Kitchen 348,Independent Cafe,KwaZulu-Natal,3270408.02
Zebra Cafe 201,Grocery Chain,Free State,3169175.74


In [0]:
%sql
-- Task 6 - Are the sales representatives hitting target?

-- Each sales representative carries a monthly revenue target. Return every sales representative with their territory, their monthly target, their actual FY2025 monthly average, and that average as a percentage of target. Sort by percentage of target, best first. Every sales representative must appear, including any who generated no revenue. 


SELECT 
    r.rep_id,
    r.rep_name,
    r.territory,
    r.target_monthly,
    ROUND((SUM(CASE
            WHEN so.order_date >= '2024-07-01' AND so.order_date <= '2025-06-30' 
                THEN sl.line_revenue
            ELSE 0
        END
        ) / 12),2) AS fy2025_avg_monthly_revenue,
    ROUND(((SUM(CASE
            WHEN so.order_date >= '2024-07-01' AND so.order_date <= '2025-06-30' 
                THEN sl.line_revenue
            ELSE 0
        END
        ) / 12) / r.target_monthly * 100),2) AS fy2025_avg_monthly_revenue_percent
FROM sizwe_reps r
LEFT JOIN sizwe_sales_orders so 
    ON r.rep_id = so.rep_id
LEFT JOIN sizwe_sales_lines sl 
    ON so.order_id = sl.order_id 
GROUP BY 1, 2, 3, 4
ORDER BY 6 DESC






rep_id,rep_name,territory,target_monthly,fy2025_avg_monthly_revenue,fy2025_avg_monthly_revenue_percent
R-016,Karabo van Wyk,Gauteng South,900000,2252880.96,250.32
R-001,Kagiso Mkhize,Gauteng North,2000000,4996325.67,249.82
R-018,Johan Maluleke,KwaZulu-Natal,900000,2184885.21,242.77
R-010,Anele Naidoo,Western Cape,900000,2011837.24,223.54
R-007,Anele Zwane,Northern Region,900000,1846514.09,205.17
R-009,Karabo Mokoena,Gauteng South,1200000,2365324.98,197.11
R-012,Anele Ismail,Eastern Cape,1200000,2299147.69,191.6
R-013,Chantal Ismail,Central Region,1200000,2285301.19,190.44
R-006,Pieter Pillay,Central Region,1200000,2068510.95,172.38
R-002,Nomsa Botha,Gauteng South,1500000,2377843.54,158.52


In [0]:
%sql
-- Task 7 - The shape of the year

-- Return monthly revenue across the full two years, with the percentage change on the previous month and a rolling three-month average alongside it. In two sentences: is there a seasonal pattern, and when is Sizwe's quietest month?

WITH monthly_revenue AS 
    (
    SELECT 
        DATE_FORMAT(so.order_date, 'yyyy-MM') AS year_month,
        ROUND(SUM(sl.line_revenue),2) AS month_revenue
    FROM sizwe_sales_orders so
    JOIN sizwe_sales_lines sl
        ON so.order_id = sl.order_id
    WHERE so.order_date BETWEEN '2023-07-01' AND '2025-06-30'
    GROUP BY 1
    ORDER BY 1 ASC
    ),

monthly_revenue_lags AS
    (
    SELECT
        year_month,
        month_revenue,
        LAG(month_revenue) OVER(ORDER BY year_month) AS month_lag_1,
        LAG(LAG(month_revenue) OVER(ORDER BY year_month)) OVER(ORDER BY year_month) AS month_lag_2,
        LAG(LAG(LAG(month_revenue) OVER(ORDER BY year_month)) OVER(ORDER BY year_month)) OVER(ORDER BY year_month) AS month_lag_3
    FROM monthly_revenue
    )

SELECT 
    year_month,
    month_revenue,
    ROUND(((month_revenue - month_lag_1) / month_lag_1 * 100), 2) AS percent_change,
    ROUND(((month_lag_1 + month_lag_2 + month_lag_3) / 3),2) AS rolling_3_month_avg,
    month_lag_1,
    month_lag_2,
    month_lag_3
FROM monthly_revenue_lags

-- Answer: It is clear that the monthly revenue drops month-over-month during the winter, more specifically from Jul to Jan (i.e. winter) and it grows from Feb to Jun (i.e. summer). The questest month was Dec 2024 but Dec 2025 also performed poorly. 


year_month,month_revenue,percent_change,rolling_3_month_avg,month_lag_1,month_lag_2,month_lag_3
2023-07,4.105147175E7,null,null,null,null,null
2023-08,3.960774434E7,-3.52,null,4.105147175E7,null,null
2023-09,3.680777906E7,-7.07,null,3.960774434E7,4.105147175E7,null
2023-10,3.578480569E7,-2.78,3.915566505E7,3.680777906E7,3.960774434E7,4.105147175E7
2023-11,3.437016262E7,-3.95,3.74001097E7,3.578480569E7,3.680777906E7,3.960774434E7
2023-12,2.987269978E7,-13.09,3.565424912E7,3.437016262E7,3.578480569E7,3.680777906E7
2024-01,3.125562108E7,4.63,3.334255603E7,2.987269978E7,3.437016262E7,3.578480569E7
2024-02,3.515602563E7,12.48,3.183282783E7,3.125562108E7,2.987269978E7,3.437016262E7
2024-03,3.728749758E7,6.06,3.209478216E7,3.515602563E7,3.125562108E7,2.987269978E7
2024-04,3.795072875E7,1.78,3.456638143E7,3.728749758E7,3.515602563E7,3.125562108E7


In [0]:
%sql
-- Task 8 - Why has the margin moved?

-- Gross margin fell from 34.7% in FY2024 to 33.0% in FY2025, while revenue grew nearly 9%. Nobody can explain it. For each product line, return revenue and gross margin for each financial year, plus that line's share of total revenue in that year. Then answer, in a short paragraph: what caused the margin decline? Your answer must point at specific numbers in your result. 

WITH revenue_cogs AS 
    (
        SELECT 
            p.product_line,
            ROUND(SUM(CASE
                    WHEN so.order_date BETWEEN '2023-07-01' AND '2024-06-30'
                        THEN sl.line_revenue
                    ELSE 0
                END),2) AS fy2024_rev,
            ROUND(SUM(CASE
                    WHEN so.order_date BETWEEN '2024-07-01' AND '2025-06-30'
                        THEN sl.line_revenue
                    ELSE 0
                END),2) AS fy2025_rev,
            ROUND(SUM(CASE
                    WHEN so.order_date BETWEEN '2023-07-01' AND '2024-06-30'
                        THEN sl.cogs
                    ELSE 0
                END),2) AS fy2024_cogs,
            ROUND(SUM(CASE
                    WHEN so.order_date BETWEEN '2024-07-01' AND '2025-06-30'
                        THEN sl.cogs
                    ELSE 0
                END),2) AS fy2025_cogs
        FROM sizwe_sales_lines sl
        JOIN sizwe_sales_orders so 
            ON sl.order_id = so.order_id
        JOIN sizwe_products p
            ON p.sku = sl.sku
        GROUP BY p.product_line
    )

SELECT 
    product_line,
    fy2024_rev,
    ROUND((100 * (fy2024_rev - fy2024_cogs) / fy2024_rev),2) AS fy2024_gm,
    ROUND((100 * fy2024_rev / SUM(fy2024_rev) OVER()),2) AS fy2024_share_percent,
    fy2025_rev,
    ROUND((100 * (fy2025_rev - fy2025_cogs) / fy2025_rev),2) AS fy2025_gm,
    ROUND((100 * fy2025_rev / SUM(fy2025_rev) OVER()),2) AS fy2025_share_percent
FROM revenue_cogs

-- Answer: The gross margin for each product line hasn't changed significantly from 2024 to 2025. However, there's been more significant changes in each of their share of the total revenue in each year. Most notably, Bulk commercial product line has increased from 25.02% to 33.65%. Bulk commercial has the smallest gross margin compared to other product lines. This contributed to the reduced gross margin in 2025. At the same time, the decline in gross margin is further accounted for by the decreased share of the total revenue for the product lines with higher gross margins. For example, Single Origin product line which has the highest gross margin has contributed relately less to the total revenue in 2025 (17.57%) compared to in 2024 (19.45%). The same reason applies to all the remaining product lines with higher gross margins.


product_line,fy2024_rev,fy2024_gm,fy2024_share_percent,fy2025_rev,fy2025_gm,fy2025_share_percent
House Blend,1.4174647148E8,38.32,32.14,1.3778447742E8,38.34,28.71
Decaf,4.630047747E7,35.85,10.5,4.434206761E7,35.81,9.24
Capsules,5.68223495E7,39.01,12.88,5.205272672E7,38.79,10.84
Bulk Commercial,1.1033858135E8,19.89,25.02,1.6150024966E8,19.91,33.65
Single Origin,8.579211767E7,44.24,19.45,8.432050194E7,44.31,17.57
